<a href="https://colab.research.google.com/github/Mohitkumar-code7/AI-Resume-Ranker/blob/main/ai_resume_ranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn sentence-transformers pandas

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


In [3]:
data = {
    "Name": ["Resume A", "Resume B", "Resume C"],
    "Text": [
        # Resume A – same meaning, different words
        "Developed intelligent systems using neural networks and data-driven models",

        # Resume B – keyword stuffing but wrong domain
        "Python Python Python Excel MS Word typing",

        # Resume C – unrelated
        "Sales marketing customer relationship management"
    ]
}

df = pd.DataFrame(data)

job_text = "“Looking for an AI engineer with experience in machine learning and deep learning"


In [4]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df["Text"].tolist() + [job_text]
)

tfidf_scores = cosine_similarity(
    tfidf_matrix[-1],
    tfidf_matrix[:-1]
)[0]

df_tfidf = df.copy()
df_tfidf["TF-IDF Score"] = tfidf_scores

df_tfidf.sort_values("TF-IDF Score", ascending=False)


,Name,Text,TF-IDF Score
0,Resume A,Developed intelligent systems using neural net...,0.0
1,Resume B,Python Python Python Excel MS Word typing,0.0
2,Resume C,Sales marketing customer relationship management,0.0


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

resume_embeddings = model.encode(df["Text"].tolist())
job_embedding = model.encode([job_text])

sbert_scores = cosine_similarity(
    job_embedding,
    resume_embeddings
)[0]

df_sbert = df.copy()
df_sbert["SBERT Score"] = sbert_scores

df_sbert.sort_values("SBERT Score", ascending=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Name,Text,SBERT Score
0,Resume A,Developed intelligent systems using neural net...,0.223006
2,Resume C,Sales marketing customer relationship management,0.032354
1,Resume B,Python Python Python Excel MS Word typing,0.006682


In [6]:
comparison = df[["Name"]].copy()
comparison["TF-IDF Score"] = df_tfidf["TF-IDF Score"]
comparison["SBERT Score"] = df_sbert["SBERT Score"]

comparison


,Name,TF-IDF Score,SBERT Score
0,Resume A,0.0,0.223006
1,Resume B,0.0,0.006682
2,Resume C,0.0,0.032354
